# Detección de plantas invasoras — Cundinamarca
Entrenamiento y evaluación de **RT-DETR** (formato YOLOv8-OBB) y **RF-DETR** (formato COCO).

Ejecuta las celdas en orden. Cada sección es independiente — puedes correr solo RT-DETR o solo RF-DETR.

## 0 · Instalación de dependencias

In [7]:
%pip install -q ultralytics
%pip install -q "rfdetr[train,loggers]"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 1 · Verificar GPU

In [ ]:
import torch
from pathlib import Path

# Rutas absolutas — funcionan sin importar desde dónde se ejecute el notebook
ROOT          = Path.cwd()
COCO_DIR      = ROOT / "Plantas.coco"
YOLO_DIR      = ROOT / "Plantas.yolov8-obb"
YOLO_YAML     = YOLO_DIR / "data.yaml"
RUNS_DIR      = ROOT / "runs"
RTDETR_RUN    = RUNS_DIR / "rtdetr" / "plantas_invasoras"
RFDETR_RUN    = RUNS_DIR / "rfdetr" / "plantas_invasoras"

# Que ultralytics resuelva `path:` del data.yaml respecto a la carpeta del proyecto
try:
    from ultralytics import settings as _ul_settings
    _ul_settings.update({"datasets_dir": str(ROOT)})
except Exception as _e:
    print(f"(aviso) no se pudo actualizar datasets_dir de ultralytics: {_e}")

print(f"ROOT: {ROOT}")
print(f"COCO_DIR existe:   {COCO_DIR.exists()}")
print(f"YOLO_YAML existe:  {YOLO_YAML.exists()}")

if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print("GPU no detectada — entrenando en CPU (AMD no tiene soporte CUDA en Windows)")

## 2 · Preparar dataset RF-DETR (COCO)
Corrige el bug de Roboflow (bbox como strings) y crea el split de validación si no existe.

In [ ]:
import json, shutil, random
from pathlib import Path

def fix_bbox_types(json_path: Path):
    with open(json_path) as f:
        coco = json.load(f)
    for ann in coco["annotations"]:
        ann["bbox"] = [float(v) for v in ann["bbox"]]
    with open(json_path, "w") as f:
        json.dump(coco, f)

def fix_all_annotations(dataset_dir: Path):
    for split in ("train", "valid", "test"):
        path = dataset_dir / split / "_annotations.coco.json"
        if path.exists():
            fix_bbox_types(path)
            print(f"Anotaciones corregidas: {path}")

def crear_split_valid(dataset_dir: Path, val_ratio=0.2, seed=42):
    train_dir = dataset_dir / "train"
    valid_dir = dataset_dir / "valid"
    ann_path  = train_dir / "_annotations.coco.json"

    if (valid_dir / "_annotations.coco.json").exists():
        print("Split de validación ya existe, nada que hacer.")
        return

    valid_dir.mkdir(parents=True, exist_ok=True)
    with open(ann_path) as f:
        coco = json.load(f)

    random.seed(seed)
    all_imgs = coco["images"][:]
    random.shuffle(all_imgs)
    n_val      = max(1, int(len(all_imgs) * val_ratio))
    val_imgs   = all_imgs[:n_val]
    train_imgs = all_imgs[n_val:]
    val_ids    = {img["id"] for img in val_imgs}

    val_anns   = [a for a in coco["annotations"] if a["image_id"] in val_ids]
    train_anns = [a for a in coco["annotations"] if a["image_id"] not in val_ids]

    for img in val_imgs:
        src = train_dir / img["file_name"]
        dst = valid_dir / img["file_name"]
        if src.exists():
            shutil.copy2(src, dst)

    with open(valid_dir / "_annotations.coco.json", "w") as f:
        json.dump({**coco, "images": val_imgs, "annotations": val_anns}, f)
    with open(ann_path, "w") as f:
        json.dump({**coco, "images": train_imgs, "annotations": train_anns}, f)

    print(f"Split creado: {len(train_imgs)} train / {len(val_imgs)} valid")

assert COCO_DIR.exists(), f"No existe {COCO_DIR}. Revisa SETUP.md."
fix_all_annotations(COCO_DIR)
crear_split_valid(COCO_DIR)

## 3 · Entrenamiento RT-DETR
Modelo transformer de Baidu. Usa el dataset en formato **YOLOv8-OBB**.

> Tarda varios minutos por época en CPU.

In [ ]:
from ultralytics import RTDETR

assert YOLO_YAML.exists(), f"No existe {YOLO_YAML}. Revisa SETUP.md."

model_rtdetr = RTDETR("rtdetr-x.pt")

results_rtdetr = model_rtdetr.train(
    data=str(YOLO_YAML),
    epochs=15,
    imgsz=640,
    batch=4,
    device="cpu",
    freeze=1,
    lr0=1e-4,
    lrf=0.01,
    warmup_epochs=3,
    patience=10,
    save=True,
    project=str(RUNS_DIR / "rtdetr"),
    name="plantas_invasoras",
    exist_ok=True,
    verbose=True,
)

print("\n--- Entrenamiento RT-DETR finalizado ---")
print(f"Mejor modelo: {RTDETR_RUN / 'weights' / 'best.pt'}")

## 4 · Entrenamiento RF-DETR
Modelo de Roboflow basado en DINOv2. Usa el dataset en formato **COCO**.

> Tarda varios minutos por época en CPU.

In [ ]:
from rfdetr import RFDETRBase

assert COCO_DIR.exists(), f"No existe {COCO_DIR}. Revisa SETUP.md."

model_rfdetr = RFDETRBase()

model_rfdetr.train(
    dataset_dir=str(COCO_DIR),
    epochs=15,
    batch_size=4,
    grad_accumulation_steps=4,
    lr=1e-4,
    num_workers=0,
    output_dir=str(RFDETR_RUN),
)

print("\n--- Entrenamiento RF-DETR finalizado ---")
print(f"Modelo guardado en: {RFDETR_RUN}")

## 5 · Evaluación y comparación
Carga los mejores pesos de cada modelo y muestra tabla comparativa.

In [ ]:
from ultralytics import RTDETR

RTDETR_WEIGHTS = RTDETR_RUN / "weights" / "best.pt"
RFDETR_METRICS = RFDETR_RUN / "metrics.csv"

resultados = []

# --- RT-DETR ---
if RTDETR_WEIGHTS.exists():
    m = RTDETR(str(RTDETR_WEIGHTS))
    met = m.val(data=str(YOLO_YAML))
    resultados.append({
        "Modelo":    "RT-DETR",
        "mAP50":     round(float(met.box.map50), 4),
        "mAP50-95":  round(float(met.box.map),   4),
        "Precisión": round(float(met.box.mp),    4),
        "Recall":    round(float(met.box.mr),    4),
    })
else:
    print(f"RT-DETR: no se encontró {RTDETR_WEIGHTS}")

# --- RF-DETR ---
# RFDETRBase no expone .val(); leemos las métricas de validación que
# Lightning va guardando en metrics.csv durante el entrenamiento y
# tomamos la mejor época por mAP50-95.
if RFDETR_METRICS.exists():
    import pandas as pd
    df = pd.read_csv(RFDETR_METRICS)
    val_cols = ["val/mAP_50", "val/mAP_50_95", "val/precision", "val/recall"]
    val_df = df.dropna(subset=[c for c in val_cols if c in df.columns], how="all")
    if len(val_df) and "val/mAP_50_95" in val_df.columns:
        best = val_df.loc[val_df["val/mAP_50_95"].idxmax()]
        resultados.append({
            "Modelo":    "RF-DETR",
            "mAP50":     round(float(best.get("val/mAP_50",    0) or 0), 4),
            "mAP50-95":  round(float(best.get("val/mAP_50_95", 0) or 0), 4),
            "Precisión": round(float(best.get("val/precision", 0) or 0), 4),
            "Recall":    round(float(best.get("val/recall",    0) or 0), 4),
        })
    else:
        print(f"RF-DETR: {RFDETR_METRICS} no tiene filas de validación todavía.")
else:
    print(f"RF-DETR: no se encontró {RFDETR_METRICS}")

# --- Tabla ---
if resultados:
    header = f"{'Modelo':<12} {'mAP50':>8} {'mAP50-95':>10} {'Precisión':>10} {'Recall':>8}"
    sep    = "-" * len(header)
    print(f"\n{sep}\n{header}\n{sep}")
    for r in resultados:
        print(f"{r['Modelo']:<12} {r['mAP50']:>8.4f} {r['mAP50-95']:>10.4f} {r['Precisión']:>10.4f} {r['Recall']:>8.4f}")
    print(sep)

## 6 · Inferencia sobre una imagen
Cambia `IMAGEN` por la ruta a cualquier foto para ver las detecciones.

In [ ]:
from ultralytics import RTDETR
from IPython.display import Image as IPImage

# Cambia esta ruta por la imagen que quieras probar
IMAGEN = ROOT / "Entrenamiento IyV" / "acacia_negra" / "acacia_negra_000001.jpg"

weights = RTDETR_RUN / "weights" / "best.pt"
assert weights.exists(), f"No existe {weights}. Entrena primero (sección 3)."
assert IMAGEN.exists(), f"No existe {IMAGEN}."

model = RTDETR(str(weights))
results = model.predict(str(IMAGEN), conf=0.25)
out = ROOT / "inferencia.jpg"
results[0].save(filename=str(out))

IPImage(str(out))